# Module 5 • Neural Networks for Natural Language Processing

# Lesson 33 • Transformer Decoder Models and Autoregressive Language Modeling

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 170–210 minutes  
**Execution target:** CPU

---

## Scope

This lesson introduces decoder-only Transformer models and autoregressive
language modeling. It develops causal self-attention, shifted next-token
targets, sequence loss, perplexity, decoding temperature, greedy search,
top-k sampling, nucleus sampling, repetition control, and controlled text
generation.

The executable experiment trains a compact decoder-only Transformer on a
synthetic command corpus. It requires no GPU, pretrained model, or internet
connection.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish encoder-only and decoder-only Transformers;
- explain causal language modeling;
- construct shifted input–target pairs;
- create causal attention masks;
- combine causal and padding masks;
- implement a decoder-only Transformer;
- calculate next-token cross-entropy;
- explain and compute perplexity;
- train a compact causal language model;
- perform greedy generation;
- apply temperature scaling;
- implement top-k sampling;
- implement nucleus sampling;
- detect repetition and premature stopping;
- discuss exposure bias and decoding risk;
- analyze Arabic and multilingual generation constraints.

## Table of Contents

1. Decoder-Only Transformer Models
2. Causal Language Modeling
3. Shifted Inputs and Targets
4. Causal Attention Masks
5. Padding Masks
6. Combined Masking
7. Autoregressive Factorization
8. Perplexity
9. Synthetic Language-Modeling Corpus
10. Tokenization
11. Vocabulary Construction
12. Sequence Encoding
13. Language-Model Dataset
14. Dynamic Padding
15. Positional Encoding
16. Decoder-Only Transformer
17. Shape Inspection
18. Mask Inspection
19. Next-Token Loss
20. Training Loop
21. Learning Curves
22. Perplexity Curve
23. Next-Token Prediction
24. Greedy Generation
25. Temperature Scaling
26. Top-k Sampling
27. Nucleus Sampling
28. Repetition Penalty
29. Controlled Generation
30. Comparing Decoding Strategies
31. Generation Evaluation
32. Distinct-n Diversity
33. Length Analysis
34. Failure Modes
35. Exposure Bias
36. Hallucination and Unsupported Continuation
37. Safety and Control
38. Computational Cost
39. Encoder-Only Versus Decoder-Only
40. Arabic and Multilingual Considerations
41. Reproducibility and Reporting
42. Knowledge Check
43. Exercises
44. Summary and Next Lesson

# 1. Decoder-Only Transformer Models

A decoder-only Transformer predicts one token from all previous tokens.

Common applications:

- text completion;
- dialogue generation;
- code generation;
- summarization through prompting;
- instruction following;
- open-ended language modeling.

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

architecture_comparison = pd.DataFrame(
    [
        (
            "Encoder-only",
            "bidirectional",
            "classification and understanding",
        ),
        (
            "Decoder-only",
            "causal left-to-right",
            "generation",
        ),
        (
            "Encoder–decoder",
            "bidirectional source + causal target",
            "conditional generation",
        ),
    ],
    columns=[
        "Architecture",
        "Context pattern",
        "Typical use",
    ],
)

architecture_comparison

# 2. Causal Language Modeling

A causal language model predicts the next token from the prefix.

Example:

```text
input prefix:  open the red
target token:  door
```

The model must not use future tokens when predicting the current target.

# 3. Shifted Inputs and Targets

For a token sequence:

```text
<BOS> open the red door <EOS>
```

the training pair is:

```text
input:  <BOS> open the red door
target: open  the  red door <EOS>
```

In [ ]:
example_sequence = [
    "<BOS>",
    "open",
    "the",
    "red",
    "door",
    "<EOS>",
]

pd.DataFrame(
    {
        "input_token": example_sequence[:-1],
        "target_token": example_sequence[1:],
    }
)

# 4. Causal Attention Masks

A causal mask blocks attention to future positions.

In [ ]:
def causal_mask_numpy(
    sequence_length: int,
) -> np.ndarray:
    return np.triu(
        np.ones(
            (
                sequence_length,
                sequence_length,
            ),
            dtype=bool,
        ),
        k=1,
    )


causal_example = causal_mask_numpy(
    6
)

pd.DataFrame(
    causal_example.astype(int)
)

`1` marks a blocked query–key pair.

# 5. Padding Masks

Padding masks block PAD keys so shorter sequences do not contribute artificial
context.

In [ ]:
padding_example = np.array(
    [
        [False, False, False, False, True, True],
        [False, False, False, False, False, True],
    ]
)

pd.DataFrame(
    padding_example.astype(int)
)

# 6. Combined Masking

Decoder-only models use:

- a causal mask shared across the batch;
- a padding mask specific to each sequence.

In [ ]:
mask_summary = pd.DataFrame(
    [
        ("Causal mask", "(T, T)", "future positions"),
        ("Padding mask", "(B, T)", "padded keys"),
    ],
    columns=["Mask", "Shape", "Blocks"],
)

mask_summary

# 7. Autoregressive Factorization

A sequence probability is factorized as:

\[
P(x_1,\dots,x_T)
=
\prod_{t=1}^{T}
P(x_t \mid x_{<t})
\]

Generation repeatedly samples or selects from the next-token distribution.

# 8. Perplexity

Perplexity is the exponential of average token cross-entropy:

\[
Perplexity = exp(loss)
\]

Lower perplexity indicates better predictive fit on the evaluated corpus.

In [ ]:
example_losses = np.array(
    [0.5, 1.0, 2.0, 3.0]
)

pd.DataFrame(
    {
        "cross_entropy": example_losses,
        "perplexity": np.exp(
            example_losses
        ),
    }
)

Perplexity values are only comparable under the same tokenization and
evaluation setup.

# 9. Synthetic Language-Modeling Corpus

The corpus contains structured commands and short domain sentences.

In [ ]:
verbs = [
    "open",
    "close",
    "find",
    "take",
    "move",
    "check",
]

modifiers = [
    "red",
    "blue",
    "green",
    "small",
    "large",
    "nearby",
]

objects = [
    "door",
    "window",
    "box",
    "book",
    "key",
    "file",
]

places = [
    "room",
    "office",
    "hall",
    "desk",
    "shelf",
    "table",
]

command_sentences = []

for verb in verbs:
    for modifier in modifiers:
        for obj in objects:
            command_sentences.append(
                f"{verb} the {modifier} {obj}"
            )

descriptive_sentences = [
    "the red door is near the office",
    "the blue box is on the table",
    "the small key is under the book",
    "the green file is on the desk",
    "the large window faces the hall",
    "the nearby shelf contains the book",
    "the doctor checks the patient record",
    "the bank approves the customer loan",
    "the server stores the application data",
    "the airport confirms the flight time",
    "the clinic updates the treatment plan",
    "the network reports a software error",
]

corpus = (
    command_sentences
    + descriptive_sentences * 6
)

corpus_frame = pd.DataFrame(
    {"text": corpus}
)

print(
    "Training sentences:",
    len(corpus_frame),
)
corpus_frame.sample(
    8,
    random_state=42,
)

# 10. Tokenization

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(
    text: str,
) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


tokenize(
    "Open the small blue window."
)

# 11. Vocabulary Construction

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

SPECIAL_TOKENS = [
    PAD_TOKEN,
    UNK_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
]

token_counts = Counter(
    token
    for text in corpus_frame[
        "text"
    ]
    for token in tokenize(text)
)

vocabulary = (
    SPECIAL_TOKENS
    + sorted(token_counts)
)

token_to_index = {
    token: index
    for index, token
    in enumerate(vocabulary)
}

PAD_ID = token_to_index[
    PAD_TOKEN
]
UNK_ID = token_to_index[
    UNK_TOKEN
]
BOS_ID = token_to_index[
    BOS_TOKEN
]
EOS_ID = token_to_index[
    EOS_TOKEN
]

print(
    "Vocabulary size:",
    len(vocabulary),
)

# 12. Sequence Encoding

In [ ]:
MAX_LENGTH = 12


def encode_text(
    text: str,
    maximum_length: int = MAX_LENGTH,
) -> list[int]:
    content_ids = [
        token_to_index.get(
            token,
            UNK_ID,
        )
        for token in tokenize(text)
    ]

    return (
        [BOS_ID]
        + content_ids[
            :maximum_length - 2
        ]
        + [EOS_ID]
    )


encoded_example = encode_text(
    "open the red door"
)

print(encoded_example)
print(
    [
        vocabulary[token_id]
        for token_id in encoded_example
    ]
)

# 13. Language-Model Dataset

In [ ]:
class LanguageModelDataset(
    Dataset
):
    def __init__(
        self,
        texts,
    ):
        self.examples = [
            torch.tensor(
                encode_text(text),
                dtype=torch.long,
            )
            for text in texts
        ]

    def __len__(self):
        return len(
            self.examples
        )

    def __getitem__(
        self,
        index,
    ):
        return self.examples[
            index
        ]


language_model_dataset = (
    LanguageModelDataset(
        corpus_frame["text"]
    )
)

len(language_model_dataset)

# 14. Dynamic Padding

In [ ]:
def collate_language_model_batch(
    batch,
):
    maximum = max(
        len(sequence)
        for sequence in batch
    )

    padded = torch.full(
        (
            len(batch),
            maximum,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    for row, sequence in enumerate(
        batch
    ):
        padded[
            row,
            :len(sequence),
        ] = sequence

    input_ids = padded[
        :,
        :-1,
    ]

    target_ids = padded[
        :,
        1:,
    ]

    return {
        "input_ids": input_ids,
        "target_ids": target_ids,
        "padding_mask": (
            input_ids == PAD_ID
        ),
    }


language_model_loader = DataLoader(
    language_model_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=(
        collate_language_model_batch
    ),
    generator=torch.Generator().manual_seed(
        42
    ),
)

sample_batch = next(
    iter(language_model_loader)
)

print(
    "Input shape:",
    sample_batch[
        "input_ids"
    ].shape,
)
print(
    "Target shape:",
    sample_batch[
        "target_ids"
    ].shape,
)

# 15. Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(
    nn.Module
):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 128,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[
            :,
            0::2,
        ] = torch.sin(
            positions * rates
        )

        encoding[
            :,
            1::2,
        ] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )

# 16. Decoder-Only Transformer

PyTorch's Transformer encoder block can implement decoder-only causal
self-attention when supplied with a causal mask.

In [ ]:
class DecoderOnlyTransformer(
    nn.Module
):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 48,
        head_count: int = 4,
        feed_forward_dimension: int = 96,
        layer_count: int = 2,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.model_dimension = (
            model_dimension
        )

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = (
            SinusoidalPositionalEncoding(
                model_dimension,
                maximum_length=MAX_LENGTH,
            )
        )

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=(
                feed_forward_dimension
            ),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = (
            nn.TransformerEncoder(
                layer,
                num_layers=layer_count,
            )
        )

        self.output_layer = nn.Linear(
            model_dimension,
            vocabulary_size,
        )

    def create_causal_mask(
        self,
        sequence_length: int,
        device: torch.device,
    ) -> torch.Tensor:
        return torch.triu(
            torch.ones(
                (
                    sequence_length,
                    sequence_length,
                ),
                dtype=torch.bool,
                device=device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        embeddings = (
            self.embedding(
                input_ids
            )
            * math.sqrt(
                self.model_dimension
            )
        )

        embeddings = self.position(
            embeddings
        )

        causal_mask = (
            self.create_causal_mask(
                sequence_length=(
                    input_ids.size(1)
                ),
                device=input_ids.device,
            )
        )

        hidden_states = (
            self.transformer(
                embeddings,
                mask=causal_mask,
                src_key_padding_mask=(
                    padding_mask
                ),
            )
        )

        logits = self.output_layer(
            hidden_states
        )

        return {
            "logits": logits,
            "hidden_states": (
                hidden_states
            ),
            "causal_mask": (
                causal_mask
            ),
        }


DEVICE = torch.device(
    "cpu"
)

torch.manual_seed(42)

model = DecoderOnlyTransformer(
    vocabulary_size=len(
        vocabulary
    ),
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter
        in model.parameters()
    ),
)

# 17. Shape Inspection

In [ ]:
with torch.no_grad():
    shape_output = model(
        sample_batch[
            "input_ids"
        ].to(DEVICE),
        sample_batch[
            "padding_mask"
        ].to(DEVICE),
    )

print(
    "Hidden states:",
    shape_output[
        "hidden_states"
    ].shape,
)
print(
    "Logits:",
    shape_output[
        "logits"
    ].shape,
)
print(
    "Causal mask:",
    shape_output[
        "causal_mask"
    ].shape,
)

# 18. Mask Inspection

In [ ]:
pd.DataFrame(
    shape_output[
        "causal_mask"
    ].int().cpu().numpy()
)

# 19. Next-Token Loss

In [ ]:
loss_function = (
    nn.CrossEntropyLoss(
        ignore_index=PAD_ID
    )
)


def language_model_loss(
    logits: torch.Tensor,
    target_ids: torch.Tensor,
) -> torch.Tensor:
    return loss_function(
        logits.reshape(
            -1,
            logits.size(-1),
        ),
        target_ids.reshape(-1),
    )


initial_loss = language_model_loss(
    shape_output["logits"],
    sample_batch[
        "target_ids"
    ].to(DEVICE),
)

print(
    "Initial loss:",
    float(initial_loss),
)
print(
    "Initial perplexity:",
    math.exp(
        min(
            float(initial_loss),
            20.0,
        )
    ),
)

# 20. Training Loop

In [ ]:
def set_seed(
    seed: int = 42,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_language_model(
    model: nn.Module,
    loader: DataLoader,
    epochs: int = 34,
    learning_rate: float = 0.003,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    history = []

    for epoch in range(epochs):
        model.train()

        losses = []
        accuracies = []
        gradient_norms = []

        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(DEVICE)

            target_ids = batch[
                "target_ids"
            ].to(DEVICE)

            padding_mask = batch[
                "padding_mask"
            ].to(DEVICE)

            optimizer.zero_grad()

            output = model(
                input_ids,
                padding_mask,
            )

            loss = language_model_loss(
                output["logits"],
                target_ids,
            )

            loss.backward()

            gradient_norm = (
                clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )
            )

            optimizer.step()

            valid = (
                target_ids != PAD_ID
            )

            predictions = (
                output["logits"]
                .argmax(dim=-1)
            )

            accuracy = (
                (
                    predictions[valid]
                    == target_ids[valid]
                )
                .float()
                .mean()
                .item()
            )

            losses.append(
                float(loss.item())
            )
            accuracies.append(
                float(accuracy)
            )
            gradient_norms.append(
                float(gradient_norm)
            )

        mean_loss = float(
            np.mean(losses)
        )

        history.append(
            {
                "epoch": epoch,
                "loss": mean_loss,
                "perplexity": math.exp(
                    min(
                        mean_loss,
                        20.0,
                    )
                ),
                "token_accuracy": float(
                    np.mean(
                        accuracies
                    )
                ),
                "gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

trained_model, training_history = (
    train_language_model(
        model,
        language_model_loader,
    )
)

print(
    "Final loss:",
    round(
        training_history[
            "loss"
        ].iloc[-1],
        4,
    ),
)
print(
    "Final token accuracy:",
    round(
        training_history[
            "token_accuracy"
        ].iloc[-1],
        3,
    ),
)

# 21. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["loss"],
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Causal Language-Modeling Loss")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "token_accuracy"
    ],
)
plt.xlabel("Epoch")
plt.ylabel("Next-token accuracy")
plt.title("Next-Token Prediction Accuracy")
plt.tight_layout()
plt.show()

# 22. Perplexity Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "perplexity"
    ],
)
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.title("Language-Model Perplexity")
plt.tight_layout()
plt.show()

# 23. Next-Token Prediction

In [ ]:
def encode_prefix(
    text: str,
) -> list[int]:
    return [
        BOS_ID
    ] + [
        token_to_index.get(
            token,
            UNK_ID,
        )
        for token in tokenize(
            text
        )
    ]


def next_token_distribution(
    model: DecoderOnlyTransformer,
    prefix: str,
) -> torch.Tensor:
    token_ids = encode_prefix(
        prefix
    )

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    padding_mask = torch.zeros_like(
        input_ids,
        dtype=torch.bool,
    )

    model.eval()

    with torch.no_grad():
        output = model(
            input_ids,
            padding_mask,
        )

        probabilities = torch.softmax(
            output["logits"][
                0,
                -1,
            ],
            dim=0,
        )

    return probabilities


probabilities = next_token_distribution(
    trained_model,
    "open the red",
)

values, indices = torch.topk(
    probabilities,
    k=5,
)

pd.DataFrame(
    {
        "token": [
            vocabulary[index]
            for index
            in indices.tolist()
        ],
        "probability": (
            values.tolist()
        ),
    }
)

# 24. Greedy Generation

Greedy decoding always selects the highest-probability next token.

In [ ]:
def greedy_generate(
    model: DecoderOnlyTransformer,
    prompt: str,
    maximum_new_tokens: int = 8,
) -> str:
    generated = encode_prefix(
        prompt
    )

    model.eval()

    with torch.no_grad():
        for _ in range(
            maximum_new_tokens
        ):
            input_ids = torch.tensor(
                [generated],
                dtype=torch.long,
                device=DEVICE,
            )

            padding_mask = (
                torch.zeros_like(
                    input_ids,
                    dtype=torch.bool,
                )
            )

            output = model(
                input_ids,
                padding_mask,
            )

            next_id = int(
                output["logits"][
                    0,
                    -1,
                ]
                .argmax()
                .item()
            )

            generated.append(
                next_id
            )

            if next_id == EOS_ID:
                break

    decoded = [
        vocabulary[token_id]
        for token_id in generated
        if token_id not in {
            BOS_ID,
            EOS_ID,
            PAD_ID,
        }
    ]

    return " ".join(decoded)


greedy_generate(
    trained_model,
    "open the",
)

# 25. Temperature Scaling

Temperature modifies logits before softmax:

\[
softmax(logits / temperature)
\]

- temperature below 1 sharpens the distribution;
- temperature above 1 flattens the distribution.

In [ ]:
example_logits = torch.tensor(
    [3.0, 2.0, 1.0]
)

temperature_rows = []

for temperature in [
    0.5,
    1.0,
    1.5,
]:
    probabilities = torch.softmax(
        example_logits
        / temperature,
        dim=0,
    )

    temperature_rows.append(
        {
            "temperature": temperature,
            "max_probability": float(
                probabilities.max()
            ),
            "entropy": float(
                -torch.sum(
                    probabilities
                    * torch.log(
                        probabilities
                        + 1e-12
                    )
                )
            ),
        }
    )

pd.DataFrame(
    temperature_rows
)

# 26. Top-k Sampling

Top-k sampling keeps only the `k` highest-probability candidates.

In [ ]:
def sample_top_k(
    logits: torch.Tensor,
    k: int,
    generator: torch.Generator,
) -> int:
    k = min(
        k,
        logits.numel(),
    )

    values, indices = torch.topk(
        logits,
        k=k,
    )

    probabilities = torch.softmax(
        values,
        dim=0,
    )

    selected = torch.multinomial(
        probabilities,
        num_samples=1,
        generator=generator,
    )

    return int(
        indices[selected].item()
    )

# 27. Nucleus Sampling

Nucleus sampling retains the smallest set of tokens whose cumulative
probability reaches threshold `p`.

In [ ]:
def sample_top_p(
    logits: torch.Tensor,
    p: float,
    generator: torch.Generator,
) -> int:
    probabilities = torch.softmax(
        logits,
        dim=0,
    )

    sorted_probabilities, sorted_indices = (
        torch.sort(
            probabilities,
            descending=True,
        )
    )

    cumulative = torch.cumsum(
        sorted_probabilities,
        dim=0,
    )

    keep = cumulative <= p

    keep[0] = True

    first_above = torch.nonzero(
        cumulative >= p,
        as_tuple=False,
    )

    if len(first_above) > 0:
        keep[
            int(first_above[0])
        ] = True

    retained_probabilities = (
        sorted_probabilities[
            keep
        ]
    )

    retained_indices = (
        sorted_indices[
            keep
        ]
    )

    retained_probabilities = (
        retained_probabilities
        / retained_probabilities.sum()
    )

    selected = torch.multinomial(
        retained_probabilities,
        num_samples=1,
        generator=generator,
    )

    return int(
        retained_indices[
            selected
        ].item()
    )

# 28. Repetition Penalty

A simple repetition penalty lowers logits for tokens already generated.

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    generated_ids: list[int],
    penalty: float,
) -> torch.Tensor:
    adjusted = logits.clone()

    for token_id in set(
        generated_ids
    ):
        if adjusted[token_id] > 0:
            adjusted[token_id] /= (
                penalty
            )
        else:
            adjusted[token_id] *= (
                penalty
            )

    return adjusted

Repetition penalties are heuristics and can suppress legitimate repetition.

# 29. Controlled Generation

In [ ]:
def generate_text(
    model: DecoderOnlyTransformer,
    prompt: str,
    maximum_new_tokens: int = 8,
    strategy: str = "top_k",
    temperature: float = 1.0,
    top_k: int = 5,
    top_p: float = 0.90,
    repetition_penalty: float = 1.1,
    seed: int = 42,
) -> str:
    generator = torch.Generator(
        device="cpu"
    ).manual_seed(seed)

    generated = encode_prefix(
        prompt
    )

    model.eval()

    with torch.no_grad():
        for _ in range(
            maximum_new_tokens
        ):
            input_ids = torch.tensor(
                [generated],
                dtype=torch.long,
                device=DEVICE,
            )

            padding_mask = (
                torch.zeros_like(
                    input_ids,
                    dtype=torch.bool,
                )
            )

            output = model(
                input_ids,
                padding_mask,
            )

            logits = output["logits"][
                0,
                -1,
            ].cpu()

            logits = (
                logits
                / max(
                    temperature,
                    1e-6,
                )
            )

            logits = (
                apply_repetition_penalty(
                    logits,
                    generated,
                    repetition_penalty,
                )
            )

            if strategy == "greedy":
                next_id = int(
                    logits.argmax().item()
                )
            elif strategy == "top_k":
                next_id = sample_top_k(
                    logits,
                    k=top_k,
                    generator=generator,
                )
            elif strategy == "top_p":
                next_id = sample_top_p(
                    logits,
                    p=top_p,
                    generator=generator,
                )
            else:
                raise ValueError(
                    "Unknown generation strategy"
                )

            generated.append(
                next_id
            )

            if next_id == EOS_ID:
                break

            if len(generated) >= (
                MAX_LENGTH
            ):
                break

    decoded_tokens = [
        vocabulary[token_id]
        for token_id in generated
        if token_id not in {
            BOS_ID,
            EOS_ID,
            PAD_ID,
        }
    ]

    return " ".join(
        decoded_tokens
    )


generate_text(
    trained_model,
    "open the",
    strategy="top_k",
    seed=7,
)

# 30. Comparing Decoding Strategies

In [ ]:
comparison_rows = []

prompts = [
    "open the",
    "find the",
    "the bank",
]

strategies = [
    "greedy",
    "top_k",
    "top_p",
]

for prompt in prompts:
    for strategy in strategies:
        comparison_rows.append(
            {
                "prompt": prompt,
                "strategy": strategy,
                "generation": (
                    generate_text(
                        trained_model,
                        prompt,
                        strategy=strategy,
                        seed=11,
                    )
                ),
            }
        )

pd.DataFrame(
    comparison_rows
)

Sampling may increase diversity but also increases error risk.

# 31. Generation Evaluation

Open-ended generation cannot be summarized by one metric.

Useful checks include:

- sequence validity;
- EOS behavior;
- repetition;
- diversity;
- prompt consistency;
- factual support.

In [ ]:
generated_samples = [
    generate_text(
        trained_model,
        "open the",
        strategy="top_k",
        seed=seed,
    )
    for seed in range(10)
]

pd.Series(
    generated_samples,
    name="generation",
)

# 32. Distinct-n Diversity

In [ ]:
def distinct_n(
    texts: list[str],
    n: int,
) -> float:
    observed = []

    for text in texts:
        tokens = tokenize(text)

        observed.extend(
            tuple(
                tokens[
                    index:index + n
                ]
            )
            for index in range(
                len(tokens) - n + 1
            )
        )

    if not observed:
        return 0.0

    return (
        len(set(observed))
        / len(observed)
    )


diversity_frame = pd.DataFrame(
    [
        (
            "distinct-1",
            distinct_n(
                generated_samples,
                1,
            ),
        ),
        (
            "distinct-2",
            distinct_n(
                generated_samples,
                2,
            ),
        ),
    ],
    columns=[
        "Metric",
        "Value",
    ],
)

diversity_frame

Higher diversity does not automatically imply better quality.

# 33. Length Analysis

In [ ]:
generation_lengths = [
    len(
        tokenize(text)
    )
    for text in generated_samples
]

pd.Series(
    generation_lengths,
    name="generated_length",
).describe()

# 34. Failure Modes

Decoder-only models may produce:

- repetition;
- premature EOS;
- failure to stop;
- unsupported continuation;
- prompt drift;
- syntactic fragments;
- memorized patterns;
- unsafe or biased output.

In [ ]:
failure_modes = pd.DataFrame(
    [
        ("Repetition", "penalty, better training, decoding constraints"),
        ("Premature EOS", "length control and better data"),
        ("No EOS", "maximum length and EOS supervision"),
        ("Prompt drift", "stronger conditioning and larger context"),
        ("Hallucination", "grounding and verification"),
    ],
    columns=["Failure", "Possible response"],
)

failure_modes

# 35. Exposure Bias

Training uses true prefixes from the dataset.

Generation uses the model's own earlier outputs.

One incorrect token can change every later prediction.

# 36. Hallucination and Unsupported Continuation

A language model predicts plausible token sequences. Plausibility is not the
same as factual correctness.

Generated content should be verified when claims depend on external facts.

# 37. Safety and Control

Generation systems should consider:

- input filtering;
- output moderation;
- constrained decoding;
- retrieval grounding;
- factual verification;
- privacy;
- human review for high-stakes use.

In [ ]:
control_layers = pd.DataFrame(
    [
        ("Prompt controls", "scope and format"),
        ("Decoding controls", "temperature, top-k, top-p"),
        ("Grounding", "external evidence"),
        ("Moderation", "unsafe content"),
        ("Human review", "high-stakes validation"),
    ],
    columns=["Control", "Purpose"],
)

control_layers

# 38. Computational Cost

During training, all target positions can be processed in parallel because the
full sequence is available with a causal mask.

During generation, tokens are produced sequentially.

In [ ]:
computational_comparison = pd.DataFrame(
    [
        ("Training", "parallel across positions", "full causal matrix"),
        ("Generation", "one token at a time", "repeated forward passes"),
        ("Long context", "quadratic attention", "high memory cost"),
    ],
    columns=["Phase", "Behavior", "Cost"],
)

computational_comparison

Production decoders commonly cache previous key and value projections to avoid
recomputing the entire prefix.

# 39. Encoder-Only Versus Decoder-Only

In [ ]:
final_architecture_comparison = pd.DataFrame(
    [
        (
            "Attention direction",
            "bidirectional",
            "causal",
        ),
        (
            "Training objective",
            "masked-token prediction",
            "next-token prediction",
        ),
        (
            "Main output",
            "contextual representations",
            "token distribution",
        ),
        (
            "Typical use",
            "understanding",
            "generation",
        ),
    ],
    columns=[
        "Property",
        "Encoder-only",
        "Decoder-only",
    ],
)

final_architecture_comparison

# 40. Arabic and Multilingual Considerations

Arabic autoregressive language modeling is affected by:

- attached clitics;
- rich morphology;
- optional tashkeel;
- MSA and dialects;
- code-switching;
- right-to-left display;
- subword fragmentation.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized form",
        "Illustrative segmentation",
    ],
)

arabic_examples

Tokenization affects both sequence length and the generation unit.

For fully vocalized Arabic tasks, tashkeel must be preserved when it is part of
the target sequence. Removing it changes the language-modeling objective.

In [ ]:
arabic_generation_choices = pd.DataFrame(
    [
        ("Word", "short sequence", "large vocabulary"),
        ("Subword", "balanced coverage", "fragmented generation"),
        ("Character", "small vocabulary", "long sequence"),
        ("Morphological segment", "clitic-aware", "analyzer dependency"),
    ],
    columns=["Unit", "Benefit", "Cost"],
)

arabic_generation_choices

Multilingual decoder models should also report language balance, script
coverage, and cross-language interference.

# 41. Reproducibility and Reporting

Report:

- training corpus;
- tokenizer and vocabulary;
- maximum sequence length;
- model dimension;
- head count;
- layer count;
- feed-forward dimension;
- positional encoding;
- optimizer and learning rate;
- batch size;
- gradient clipping;
- number of epochs;
- perplexity;
- decoding strategy;
- temperature;
- top-k or top-p settings;
- repetition penalty;
- random seeds;
- hardware.

In [ ]:
import platform

metadata = pd.Series(
    {
        "training_sentences": len(
            corpus_frame
        ),
        "vocabulary_size": len(
            vocabulary
        ),
        "maximum_length": MAX_LENGTH,
        "model_dimension": 48,
        "attention_heads": 4,
        "layers": 2,
        "feed_forward_dimension": 96,
        "epochs": len(
            training_history
        ),
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": (
            platform.python_version()
        ),
        "numpy_version": (
            np.__version__
        ),
        "torch_version": (
            torch.__version__
        ),
    },
    name="Decoder-only experiment",
)

metadata

# 42. Knowledge Check

1. What is a decoder-only Transformer?
2. What is causal language modeling?
3. Why are input and target sequences shifted?
4. What does a causal mask block?
5. What does a padding mask block?
6. How is sequence probability factorized?
7. How is perplexity related to loss?
8. Why can training be parallelized across positions?
9. Why is generation sequential?
10. How does temperature affect sampling?
11. How does top-k sampling work?
12. How does nucleus sampling work?
13. What is exposure bias?
14. Why can plausible text still be false?
15. How do Arabic morphology and tashkeel affect generation?

# 43. Exercises

## Exercise 1 — Shifted Targets

Construct input and target tensors manually.

## Exercise 2 — Causal Mask

Verify that future positions receive zero attention.

## Exercise 3 — Perplexity

Compare perplexity across several corpus partitions.

## Exercise 4 — Temperature

Generate text with temperatures 0.5, 1.0, and 1.5.

## Exercise 5 — Top-k

Compare `k=2`, `k=5`, and `k=10`.

## Exercise 6 — Nucleus Sampling

Compare `p=0.7`, `p=0.9`, and `p=0.98`.

## Exercise 7 — Repetition Penalty

Measure repetition before and after applying a penalty.

## Exercise 8 — Prompt Robustness

Test incomplete, noisy, and OOV prompts.

## Exercise 9 — Arabic Generation

Build a fully vocalized Arabic command corpus.

## Exercise 10 — KV Caching

Design a cached autoregressive decoding procedure.

## Challenge Exercises

1. Tie the output projection to the input embedding table.
2. Implement label smoothing.
3. Add learning-rate warmup and cosine decay.
4. Implement key–value caching.
5. Add constrained decoding with a permitted vocabulary.

# 44. Summary and Next Lesson

In this lesson:

- decoder-only Transformers were introduced for generation;
- causal language modeling predicted each token from its prefix;
- shifted inputs and targets were constructed;
- causal and padding masks were distinguished;
- next-token cross-entropy and perplexity were calculated;
- a compact decoder-only Transformer was trained on CPU;
- greedy, temperature-controlled, top-k, and nucleus decoding were
  implemented;
- repetition penalties and generation limits were applied;
- diversity, length, and failure modes were analyzed;
- exposure bias, hallucination, safety, and control were discussed;
- Arabic morphology, segmentation, and tashkeel were connected to
  autoregressive generation.

## Next Lesson

**Lesson 34: Transformer Encoder–Decoder Models for Conditional Generation**
introduces cross-attention, source and target masks, Transformer translation,
teacher forcing, greedy decoding, beam search foundations, and sequence-level
evaluation.

# References

- Vaswani, A. et al. *Attention Is All You Need*.
- Radford, A. et al. GPT language-modeling literature.
- Holtzman, A. et al. nucleus sampling literature.
- Bengio, S. et al. exposure-bias and scheduled-sampling literature.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.